<a href="https://colab.research.google.com/github/diddat/Problem-Set-2-Econ_3916_PS2.ipynb/blob/main/Econ_3916_ps2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [ ]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.binomial(1, 0.22, size=n)
    latent = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    satisfaction = np.clip(np.round(latent), 1, 5)
    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "latent_satisfaction": latent,
        "satisfaction": satisfaction,
    })

pre = simulate_population(BASE_BEFORE)
post = simulate_population(BASE_AFTER)
print(f"True population mean satisfaction: before = 3.30, after = 3.40, true improvement = 0.10")


True population mean satisfaction: before = 3.30, after = 3.40, true improvement = 0.10


The true means follow from the bases. Engagement_z is z-scored, so its mean is 0, and 22% have a support ticket, so the latent mean is base - 0.60 x 0.22: 3.432 - 0.132 = 3.30 before, and 3.532 - 0.132 = 3.40 after. The true improvement is therefore 0.10 points.

### Step 1.2: Impose the response mechanism (15 pts)

In [ ]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    z = a + b * (df["latent_satisfaction"] - 3.3) + 0.30 * df["engagement_z"] - 0.25 * df["had_support_ticket"]
    out = df.copy()
    out["p_respond"] = 1 / (1 + np.exp(-z))
    out["responded"] = rng.random(len(df)) < out["p_respond"]
    return out

pre = respond(pre, a=-1.0, b=0.45)
post = respond(post, a=-2.75, b=2.00)
for name, df in [("Before", pre), ("After", post)]:
    resp = df[df["responded"]]
    print(f"{name}: respondents = {len(resp)}, response rate = {df['responded'].mean():.3f}, observed survey mean = {resp['satisfaction'].mean():.3f}")
obs_pre = pre[pre["responded"]]["satisfaction"].mean()
obs_post = post[post["responded"]]["satisfaction"].mean()
print(f"Apparent gap = {obs_post - obs_pre:.3f} points, out of a true improvement of 0.10 points.")


Before: respondents = 13768, response rate = 0.275, observed survey mean = 3.600
After: respondents = 7597, response rate = 0.152, observed survey mean = 4.304
Apparent gap = 0.704 points, out of a true improvement of 0.10 points.


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

This is Chapter 6 nonresponse bias, specifically missing not at random (MNAR): response depends on satisfaction itself, not only on observed covariates. Respondents are therefore a self-selected, more satisfied slice of the population.

A larger survey would not fix it. A larger sample shrinks sampling noise around the respondent mean, but the bias (about +0.30 points before and +0.90 points after) does not shrink with sample size. It would give a more precise wrong answer.

The survey that produced the better number heard from fewer people: 27.5% responded before the redesign and only 15.2% after.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [ ]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

import warnings
warnings.filterwarnings("ignore")
pre_resp = pre[pre["responded"]]["satisfaction"].to_numpy()
post_resp = post[post["responded"]]["satisfaction"].to_numpy()
for name, x in [("Before", pre_resp), ("After", post_resp)]:
    hand_mean = np.percentile(bootstrap_by_hand(x, np.mean), [2.5, 97.5])
    hand_med = np.percentile(bootstrap_by_hand(x, np.median), [2.5, 97.5])
    print(name, "mean, hand percentile CI:", np.round(hand_mean, 3))
    print(name, "median, hand percentile CI:", np.round(hand_med, 3))
    mean_pct = stats.bootstrap((x,), np.mean, n_resamples=2000, confidence_level=0.95, method="percentile", random_state=1)
    mean_bca = stats.bootstrap((x,), np.mean, n_resamples=2000, confidence_level=0.95, method="BCa", random_state=1)
    med_pct = stats.bootstrap((x,), np.median, n_resamples=2000, confidence_level=0.95, method="percentile", random_state=1)
    med_bca = stats.bootstrap((x,), np.median, n_resamples=2000, confidence_level=0.95, method="BCa", random_state=1)
    print(name, "mean, scipy percentile:", np.round([mean_pct.confidence_interval.low, mean_pct.confidence_interval.high], 3),
          " BCa:", np.round([mean_bca.confidence_interval.low, mean_bca.confidence_interval.high], 3))
    print(name, "median, scipy percentile:", np.round([med_pct.confidence_interval.low, med_pct.confidence_interval.high], 3),
          " BCa:", np.round([med_bca.confidence_interval.low, med_bca.confidence_interval.high], 3))


Before mean, hand percentile CI: [3.585 3.615]
Before median, hand percentile CI: [4. 4.]
Before mean, scipy percentile: [3.585 3.614]  BCa: [3.586 3.615]
Before median, scipy percentile: [4. 4.]  BCa: [nan nan]
After mean, hand percentile CI: [4.289 4.319]
After median, hand percentile CI: [4. 4.]
After mean, scipy percentile: [4.289 4.319]  BCa: [4.289 4.319]
After median, scipy percentile: [4. 4.]  BCa: [nan nan]


For the mean, the hand and scipy intervals agree closely. The hand percentile intervals match the scipy percentile intervals to within about 0.001, and the BCa intervals differ from the percentile intervals by less than 0.001 for both periods. BCa corrects for bias and skewness, but with 7,000-14,000 respondents the bootstrap distribution for the mean is already almost symmetric, so the correction is tiny.

For the median they differ in a different way: the percentile interval collapses to [4, 4] in both periods, and the BCa interval cannot be computed at all (scipy returns NaN). That is not a coding error. Satisfaction is an integer on a 1-5 scale and both medians are 4, so almost every bootstrap median is also 4 and there is no variation for BCa's acceleration correction to work with.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [ ]:
# 200 repeated surveys, each with a fresh population and the same response mechanism.
# A percentile bootstrap CI (B=200, for speed) is checked against the true means 3.30 / 3.40.
cov_rng = np.random.default_rng(7)
R, B = 200, 200
cover_pre = cover_post = 0
for r in range(R):
    for label, base, a, b, truth in [("pre", BASE_BEFORE, -1.0, 0.45, 3.30),
                                     ("post", BASE_AFTER, -2.75, 2.00, 3.40)]:
        df = respond(simulate_population(base, rng=cov_rng), a, b, rng=cov_rng)
        x = df[df["responded"]]["satisfaction"].to_numpy()
        idx = cov_rng.integers(0, len(x), size=(B, len(x)))
        lo, hi = np.percentile(x[idx].mean(axis=1), [2.5, 97.5])
        if lo <= truth <= hi:
            if label == "pre":
                cover_pre += 1
            else:
                cover_post += 1
for name, k in [("Before", cover_pre), ("After", cover_post)]:
    p = k / R
    mc_se = np.sqrt(p * (1 - p) / R)
    print(f"{name}: coverage = {p:.3f} ({k}/{R}), Monte Carlo SE = {mc_se:.3f}")


Before: coverage = 0.000 (0/200), Monte Carlo SE = 0.000
After: coverage = 0.000 (0/200), Monte Carlo SE = 0.000


Coverage is 0 out of 200 for both periods, with a Monte Carlo standard error of 0.000 by the usual formula sqrt(p(1-p)/R). A gap smaller than that error would not be a finding, but this gap is not small: a nominal 95% interval covering the truth 0% of the time is a failure of the interval, not of the sample size.

The reason is bias, not width. The intervals are only about 0.03 points wide, while the respondent means sit about 0.30 (before) and 0.90 (after) points above the truth. No 95% interval centred on a biased estimate can cover a truth that far away, however many bootstrap resamples are used.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [ ]:
t_stat, p_value = stats.ttest_ind(post_resp, pre_resp, equal_var=False)
n1, n2 = len(pre_resp), len(post_resp)
pooled_sd = np.sqrt(((n1 - 1) * pre_resp.var(ddof=1) + (n2 - 1) * post_resp.var(ddof=1)) / (n1 + n2 - 2))
cohens_d = (post_resp.mean() - pre_resp.mean()) / pooled_sd
print(f"Welch t = {t_stat:.2f}, p = {p_value:.3g}")
print(f"Observed difference = {post_resp.mean() - pre_resp.mean():.3f} points, pooled SD = {pooled_sd:.3f}, Cohen's d = {cohens_d:.3f}")


Welch t = 65.17, p = 0
Observed difference = 0.704 points, pooled SD = 0.815, Cohen's d = 0.864


The test is overwhelmingly significant, and the effect size is large (Cohen's d about 0.86), but that only says the two respondent samples differ far beyond sampling noise. It says nothing about whether the populations differ by that much. The gap is produced by a change in who answers, not (mainly) by a change in how satisfied users are: the post-redesign mechanism selects much more strongly on satisfaction (b = 2.00 versus 0.45) and the response rate falls. A tiny p-value cannot distinguish a real improvement from a change in selection, so significance here is answering the wrong question.

### Step 3.2: Power, before the fact

In [ ]:
# Two-sided alpha = 0.05, power = 0.80, per-group n for a true improvement of 0.10.
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
n_per_group = 2 * ((z * pooled_sd) / 0.10) ** 2
print(f"Respondents needed per group for 80% power at a 0.10-point effect: {n_per_group:.0f}")
print(f"(Using the full-population SD of about 0.89 instead gives about {2 * ((z * 0.89) / 0.10) ** 2:.0f} per group.)")
detectable = z * pooled_sd * np.sqrt(1 / n1 + 1 / n2)
print(f"At the sample sizes actually obtained ({n1} before, {n2} after), the detectable effect at 80% power is {detectable:.3f} points.")

# Would a larger sample make the estimate better? Vary the invited population size, same mechanism.
for n in [5000, 50000, 500000]:
    local_rng = np.random.default_rng(42)
    d_pre = respond(simulate_population(BASE_BEFORE, n=n, rng=local_rng), -1.0, 0.45, rng=local_rng)
    d_post = respond(simulate_population(BASE_AFTER, n=n, rng=local_rng), -2.75, 2.00, rng=local_rng)
    m_pre = d_pre[d_pre["responded"]]["satisfaction"].mean()
    m_post = d_post[d_post["responded"]]["satisfaction"].mean()
    se = np.sqrt(d_pre[d_pre["responded"]]["satisfaction"].var(ddof=1) / d_pre["responded"].sum()
                  + d_post[d_post["responded"]]["satisfaction"].var(ddof=1) / d_post["responded"].sum())
    print(f"Invited n = {n}: apparent gap = {m_post - m_pre:.3f}, SE of gap = {se:.4f}")


Respondents needed per group for 80% power at a 0.10-point effect: 1043
(Using the full-population SD of about 0.89 instead gives about 1243 per group.)
At the sample sizes actually obtained (13768 before, 7597 after), the detectable effect at 80% power is 0.033 points.
Invited n = 5000: apparent gap = 0.736, SE of gap = 0.0339
Invited n = 50000: apparent gap = 0.708, SE of gap = 0.0106
Invited n = 500000: apparent gap = 0.707, SE of gap = 0.0034


No. The simulation shows the apparent gap stays near 0.71 points (0.71 to 0.74) as the invited population grows from 5,000 to 500,000, while the standard error of the gap shrinks. A larger sample therefore makes the estimate more precise around the wrong value (about 0.71 instead of the true 0.10) and makes the misleading result look more certain. Under this response mechanism, sample size fixes variance, not bias. The fix has to change who is heard from, not how many are heard from (see the slide in Phase 4).

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [ ]:
from sklearn.linear_model import LogisticRegression

def ipw_mean(df):
    X = df[["engagement_z", "had_support_ticket"]].to_numpy()
    model = LogisticRegression(max_iter=1000).fit(X, df["responded"])
    p_hat = model.predict_proba(X)[:, 1]
    resp = df[df["responded"]]
    w = 1 / p_hat[df["responded"].to_numpy()]
    return np.sum(w * resp["satisfaction"]) / np.sum(w)

def ipw_bootstrap(df, B=200, seed=3):
    rng2 = np.random.default_rng(seed)
    vals = []
    for _ in range(B):
        boot = df.iloc[rng2.integers(0, len(df), len(df))]
        vals.append(ipw_mean(boot))
    return np.percentile(vals, [2.5, 97.5])

for name, df, truth in [("Before", pre, 3.30), ("After", post, 3.40)]:
    est = ipw_mean(df)
    lo, hi = ipw_bootstrap(df)
    obs = df[df["responded"]]["satisfaction"].mean()
    print(f"{name}: observed = {obs:.3f}, IPW = {est:.3f}, 95% bootstrap CI = [{lo:.3f}, {hi:.3f}], true = {truth:.2f}")


Before: observed = 3.600, IPW = 3.464, 95% bootstrap CI = [3.450, 3.478], true = 3.30
After: observed = 4.304, IPW = 4.143, 95% bootstrap CI = [4.123, 4.163], true = 3.40


**Your revision: what you changed and why.** Version 1 left the target unnamed and used raw engagement_minutes in a generic data frame, with no period, seed, or true mean to check against. Version 2 names the post-redesign data frame (post), uses engagement_z and had_support_ticket exactly as they appear in this notebook, sets B = 200 so the bootstrap refit runs in reasonable time, and asks whether the correction recovers the true post mean of 3.40.

**Your check: the corrected mean against your known true mean.** Before: IPW estimate 3.464 against a true mean of 3.30, still 0.16 points too high. After: IPW estimate 4.143 against a true mean of 3.40, still 0.74 points too high, with a 95% bootstrap interval of [4.123, 4.163] that does not come close to the truth. The correction did not work. That is the expected result, and it is the honest finding: weighting moves the estimate in the right direction but cannot recover the truth here.

**In your own words: how the weights are computed.** The logistic model is fitted to the response indicator using only engagement_z and had_support_ticket, so each user gets a predicted probability of responding based on those covariates alone. Each respondent is then weighted by one over that predicted probability, so respondents who look like people who rarely answer count more heavily. The weighted mean is the sum of weight times satisfaction over respondents, divided by the sum of the weights.

**Would you use AI for this step again, and why?** Yes, for drafting the weighting code and naming the assumption it rests on, but not for trusting the corrected number without checking it against a known truth first.

### Task 4.2: The slide (5 pts)

PM slide, revised

What the survey shows: among people who answered, mean satisfaction rose from 3.60 before the redesign to 4.30 after, a gap of 0.70 points.

What it does not show: that users overall became more satisfied. The survey heard from fewer people after the redesign (15% versus 28%), and answering depended strongly on satisfaction itself. The true improvement in this simulation was 0.10 points, from 3.30 to 3.40.

One number with an interval: post-redesign observed mean 4.30, 95% bootstrap interval [4.29, 4.32]. That interval covers sampling noise only, not the selection bias, so it should not be read as the range for overall satisfaction.

The one change that would fix this properly: follow up a random sample of nonrespondents, with enough effort to hear from them, so the final sample no longer depends on satisfaction itself.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Muse Spark, 2026-10-26

*Version 1 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

*Reply:*

Here is an inverse-probability weighting (IPW) correction using a logistic model of response on the covariates.

### Code

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm

# df is the sampled population, with one row per user:
#   engagement_minutes, had_support_ticket, satisfaction, responded
# responded = 1 if the user answered the survey, 0 otherwise
# satisfaction is observed only for respondents, but the covariates
# and responded must be known for everyone in the sampling frame.

def ipw_mean(data, n_boot=2000, seed=0):
    X = sm.add_constant(data[["engagement_minutes", "had_support_ticket"]])
    model = sm.Logit(data["responded"], X).fit(disp=0)
    p_hat = np.clip(model.predict(X), 0.01, 0.99)

    resp = data[data["responded"] == 1].copy()
    w = 1.0 / p_hat[data["responded"] == 1]
    weighted_mean = np.average(resp["satisfaction"], weights=w)

    rng = np.random.default_rng(seed)
    boot_means = []
    for _ in range(n_boot):
        boot = data.sample(n=len(data), replace=True, random_state=int(rng.integers(1e9)))
        try:
            Xb = sm.add_constant(boot[["engagement_minutes", "had_support_ticket"]])
            model_b = sm.Logit(boot["responded"], Xb).fit(disp=0)
            p_b = np.clip(model_b.predict(Xb), 0.01, 0.99)
            resp_b = boot[boot["responded"] == 1]
            w_b = 1.0 / p_b[boot["responded"] == 1]
            boot_means.append(np.average(resp_b["satisfaction"], weights=w_b))
        except Exception:
            continue

    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return weighted_mean, (lo, hi)

weighted_mean, (lo, hi) = ipw_mean(df)
naive_mean = df.loc[df["responded"] == 1, "satisfaction"].mean()

print(f"Naive respondent mean: {naive_mean:.3f}")
print(f"IPW weighted mean:     {weighted_mean:.3f}")
print(f"95% bootstrap CI:      [{lo:.3f}, {hi:.3f}]")
```

This is the Hajek IPW estimator: each respondent is weighted by `1 / P(respond | covariates)`, so groups that were under-represented among respondents are up-weighted to match the population's covariate distribution. The bootstrap refits the logistic model in each resample, so the interval reflects uncertainty in both the weights and the mean.

### Mechanism Check: the assumption that makes this valid

IPW is valid under Missing At Random (MAR), also called ignorability, plus positivity:

$$P(\text{respond} \mid \text{satisfaction}, \text{engagement\_minutes}, \text{had\_support\_ticket}) = P(\text{respond} \mid \text{engagement\_minutes}, \text{had\_support\_ticket})$$

and $0 < P(\text{respond} \mid \text{covariates}) < 1$ for all covariate values.

In words: once you condition on the observed covariates, response must be independent of satisfaction itself. Then reweighting on those covariates is enough, because within each covariate group respondents are a random subset with respect to satisfaction.

When response depends on satisfaction ITSELF, that assumption fails. This is Missing Not At Random (MNAR). The true response probability depends on a variable that is unobserved for non-respondents, so it cannot be estimated from the observed data: the logistic model above omits the key predictor, the estimated propensities are misspecified, and within every covariate group the respondents are still a higher-satisfaction subset than the non-respondents. No weighting on covariates alone can undo selection that happens within covariate groups.

### Evaluate: will this recover the true mean?

**No.**

Given your mechanism — where the probability of responding increases with satisfaction directly, not only through `engagement_minutes` and `had_support_ticket` — IPW on the covariates will remove only the covariate-driven part of the bias. It will rebalance engagement and support-ticket composition, so the weighted mean should move down from the naive respondent mean, but it will remain biased upward relative to the true population mean, because the within-group selection on satisfaction itself is untouched.

*Change made in version 2:* Version 1 left the target unnamed and used raw engagement_minutes in a generic data frame, with no period, seed, or true mean to check against. Version 2 names the post-redesign data frame (post), uses engagement_z and had_support_ticket exactly as they appear in this notebook, sets B = 200 so the bootstrap refit runs in reasonable time, and asks whether the correction recovers the true post mean of 3.40.

*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I have a simulated post-redesign population in a DataFrame called post, with columns engagement_z, had_support_ticket, satisfaction, and responded (True/False), seed 42 in my notebook. The probability of responding increases with satisfaction itself. Write Python that implements inverse-probability weighting to correct the POST-redesign observed survey mean, using a logistic model of response on engagement_z and had_support_ticket only (not raw engagement_minutes).

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my variable names exactly: post, engagement_z, had_support_ticket, satisfaction, responded. Return the weighted estimate and its bootstrap interval, with B=200 to keep runtime reasonable.

[Mechanism Check] Explain precisely which assumption makes this correction valid, and what happens to it when response depends on satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true post mean of 3.40? Answer yes or no and justify it.

*Reply:*

Here is inverse-probability weighting for the post-redesign mean, using only the observed covariates.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

def ipw_mean(df):
    X = df[["engagement_z", "had_support_ticket"]].to_numpy()
    responded = df["responded"].to_numpy()
    satisfaction = df["satisfaction"].to_numpy()

    model = LogisticRegression(max_iter=1000).fit(X, responded.astype(int))
    p_hat = model.predict_proba(X)[:, 1]

    # Weight respondents by 1 / P(respond | covariates)
    weights = 1 / p_hat[responded]
    return np.sum(weights * satisfaction[responded]) / np.sum(weights)

observed_mean = post.loc[post["responded"], "satisfaction"].mean()
weighted_mean = ipw_mean(post)

# Bootstrap interval, refitting the response model each time
rng = np.random.default_rng(0)
boot_means = []
for _ in range(200):
    boot = post.iloc[rng.integers(0, len(post), len(post))]
    boot_means.append(ipw_mean(boot))

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

print(f"Observed post mean: {observed_mean:.3f}")
print(f"IPW post mean: {weighted_mean:.3f}")
print(f"95% bootstrap interval: [{ci_low:.3f}, {ci_high:.3f}]")
```

### Mechanism check

This correction is valid only under missing at random (MAR) given the covariates, also called ignorability: response must be independent of satisfaction conditional on `engagement_z` and `had_support_ticket`, i.e. P(responded | satisfaction, covariates) = P(responded | covariates). It also requires positivity, so every user has a non-zero chance of responding, and a correctly specified response model.

When response depends on satisfaction itself, even after conditioning on the covariates, that assumption fails. This is missing not at random (MNAR) / nonignorable nonresponse. The fitted model estimates P(responded | covariates) only, so it misses the part of selection driven by satisfaction. Within each covariate group, more satisfied users are still more likely to respond, and weighting by the covariate-based probabilities cannot undo that within-group selection.

### Evaluate

No. Given your mechanism, this correction will not recover the true post mean of 3.40. It may move the estimate down from the observed mean, because the covariates are correlated with satisfaction, but the remaining selection on satisfaction itself leaves the weighted mean biased upward.


## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.